In [1]:
import pandas as pd

df = pd.read_json("../results/agent_evaluation_full_v2.jsonl", lines=True)
df

,image_path,image_features,triage_result,specialist_queue,trace,ground_truth,melanocytic_gt
0,../dataset/test/ISIC_0025686.jpg,{'global': {'symmetry': {'shape': 'asymmetric'...,"{'_raw': '{ ""reasoning_steps"": { ""initia...",[],"[{'agent': 'observer', 'role': 'user', 'payloa...",Nevus,True
1,../dataset/test/ISIC_0031196.jpg,"{'global': {'symmetry': {'shape': 'symmetric',...","{'_raw': '{ ""reasoning_steps"": { ""initia...",[],"[{'agent': 'observer', 'role': 'user', 'payloa...",Pigmented benign keratosis,False
2,../dataset/test/ISIC_0030535.jpg,"{'global': {'symmetry': {'shape': 'symmetric',...",{'reasoning_steps': {'initial_finding': 'No St...,[],"[{'agent': 'observer', 'role': 'user', 'payloa...",Nevus,True
3,../dataset/test/ISIC_0024994.jpg,"{'global': {'symmetry': {'shape': 'symmetric',...","{'_raw': '{ ""reasoning_steps"": { ""initia...",[],"[{'agent': 'observer', 'role': 'user', 'payloa...",Dermatofibroma,False
4,../dataset/test/ISIC_0024944.jpg,"{'global': {'symmetry': {'shape': 'symmetric',...","{'_raw': '{ ""reasoning_steps"": { ""initia...",[],"[{'agent': 'observer', 'role': 'user', 'payloa...",Nevus,True
...,...,...,...,...,...,...,...
295,../dataset/test/ISIC_0025213.jpg,{'global': {'symmetry': {'shape': 'asymmetric'...,"{'_raw': '{ ""reasoning_steps"": { ""initia...",[],"[{'agent': 'observer', 'role': 'user', 'payloa...",Nevus,True
296,../dataset/test/ISIC_0029123.jpg,{'global': {'symmetry': {'shape': 'asymmetric'...,"{'_raw': '{ ""reasoning_steps"": { ""initia...",[],"[{'agent': 'observer', 'role': 'user', 'payloa...",Basal cell carcinoma,False
297,../dataset/test/ISIC_0025904.jpg,{'global': {'symmetry': {'shape': 'asymmetric'...,"{'_raw': '{ ""reasoning_steps"": { ""initia...",[],"[{'agent': 'observer', 'role': 'user', 'payloa...",Melanoma,True
298,../dataset/test/ISIC_0025915.jpg,{'global': {'symmetry': {'shape': 'asymmetric'...,"{'_raw': '{ ""reasoning_steps"": { ""initia...",[],"[{'agent': 'observer', 'role': 'user', 'payloa...",Pigmented benign keratosis,False


In [18]:
df.loc[220, "trace"][-1]

{'agent': 'diagnosis',
 'role': 'user',
 'payload': {'_raw': '{\n  "diagnosis": "melanoma",\n  "confidence": 0.82,\n  "differential": [\n    {\n      "label": "bcc",\n      "reason": "Presence of multiple blue-gray ovoid nests/ clods — a classic BCC anchor that could represent a pigmented BCC mimic."\n    },\n    {\n      "label": "pbk",\n      "reason": "Structureless, heavily pigmented lesion with large dark blotches and white halos could reflect a pigmented benign keratosis mimic if clinical context favors a stuck-on lesion, though fewer supporting keratin features are present."\n    }\n  ],\n  "support": [\n    "Multiple blue-black globules of variable size throughout the lesion (central and widespread)",\n    "Large irregular dark blue-black blotches centrally",\n    "Central blue-gray veil-like homogeneous areas",\n    "Thin peripheral pigment rim at lesion edge",\n    "Asymmetric shape and asymmetric color distribution"\n  ],\n  "warning": [\n    "Blue-gray ovoid nests/ clods (B

In [25]:
from pathlib import Path
import json, re

def clean_raw_json(raw_str):
    cleaned = ''.join(
        c if ord(c) >= 32 and c != '\x7f' else ' '
        for c in raw_str
    )

    # Step 2: Ensure newlines inside strings are escaped as \\n
    # We do this by protecting real structural newlines (outside strings)
    # But since we don't want full parser, just escape all \n not preceded by \
    def escape_unescaped_newline(match):
        prefix = match.group(1)
        if len(prefix) % 2 == 0:  # Even backslashes → not escaped
            return f'{prefix}\\n'
        else:  # Odd → already escaping something, leave alone
            return match.group(0)
    
    # This is complex — simpler: just replace \n with \\n, but avoid \\n → \\\\n
    cleaned = re.sub(r'(?<!\\)(?:\\\\)*\n', '\\\\n', cleaned)
    cleaned = re.sub(r'(?<!\\)(?:\\\\)*\r', '\\\\r', cleaned)

    # Step 3: Make sure we didn’t lose the opening {
    stripped = cleaned.strip()
    if not stripped.startswith('{'):
        # Try to find the first '{' and cut everything before it
        pos = cleaned.find('{')
        if pos != -1:
            cleaned = cleaned[pos:]
        else:
            # Desperate: prepend {
            cleaned = '{' + cleaned

    # Step 4: Make sure it ends with }
    if not cleaned.strip().endswith('}'):
        pos = cleaned.rfind('}')
        if pos != -1:
            cleaned = cleaned[:pos+1]
        else:
            # Last resort
            cleaned = cleaned + '}'

    return cleaned

csv_payload = []
for index, item in df.iterrows():
    trace = item["trace"]
    
    dx_agent_payload = next((t for t in trace if t.get("agent") == "diagnosis"), None)
    if not dx_agent_payload:
        csv_payload.append({
            "image_path": Path(item["image_path"]).stem,
            "diagnosis": None, "confidence": None, "differential": None,
            "support": None, "warning": None, "recommendation": None,
            "ground_truth": item["ground_truth"],
            "melanocytic_gt": item["melanocytic_gt"]
        })
        continue

    payload = dx_agent_payload.get("payload", None)
    if not payload:
        # Handle missing payload
        data = {}
    elif isinstance(payload, dict) and '_raw' not in payload:
        # Normal case: already parsed
        data = payload
    elif isinstance(payload, dict) and '_raw' in payload:
        raw_json_str = payload['_raw']
        
        # Clean the problematic string concatenations
        cleaned_str = clean_raw_json(raw_json_str)

        try:
            data = json.loads(cleaned_str)
        except (json.JSONDecodeError, TypeError) as e:
            print(f"Failed to parse cleaned _raw JSON at index {index}: {e}")
            print(f"Raw (cleaned): {cleaned_str[700:750]}...")  # Print context near error
            data = {}
    else:
        data = {}

    # Extract fields safely
    diagnosis = data.get("diagnosis")
    confidence = data.get("confidence")
    differential = data.get("differential")
    support = data.get("support")
    warning = data.get("warning")
    recommendation = data.get("recommendation")
    
    csv_payload.append({
        "image_path": Path(item["image_path"]).stem,
        "diagnosis": diagnosis,
        "ground_truth": item["ground_truth"],
        "melanocytic_gt": item["melanocytic_gt"],
        "confidence": confidence,
        "differential": differential,
        "support": support,
        "warning": warning,
        "recommendation": recommendation
    })


df_payload = pd.DataFrame(csv_payload)

In [26]:
df_payload.head()

,image_path,diagnosis,ground_truth,melanocytic_gt,confidence,differential,support,warning,recommendation
0,ISIC_0025686,melanoma,Nevus,True,0.60,"[{'label': 'scc', 'reason': 'Diffuse pink-red ...","[Asymmetric shape and color, Structureless mul...",[No strong melanocytic-specific anchors presen...,Lesion is indeterminate with concerning featur...
1,ISIC_0031196,scc,Pigmented benign keratosis,False,0.65,"[{'diagnosis': 'ak', 'reason': 'Superficial wh...",[Fine superficial white scaling (keratin scale...,[Keratin-malignancy override applied: ≥2 kerat...,Consider diagnostic excision or punch/incision...
2,ISIC_0030535,nevus,Nevus,True,0.85,"[{'label': 'melanoma', 'reason': 'multiple sha...",[Symmetric shape and symmetric color distribut...,[No single high-specificity melanocytic anchor...,Consider conservative management with clinical...
3,ISIC_0024994,df,Dermatofibroma,False,0.75,"[{'diagnosis': 'nevus', 'reason': 'structurele...",[symmetric shape and symmetric color distribut...,[shiny white lines and central scar-like featu...,For informational use: correlate with clinical...
4,ISIC_0024944,nevus,Nevus,True,0.85,"[{'label': 'melanoma', 'reason': 'Irregular pa...",[Symmetric shape and symmetric color distribut...,[No classic regular pigment network centrally ...,For informational use: consider short-term pho...


In [27]:
df_payload["diagnosis"].value_counts()

diagnosis
melanoma                          120
nevus                             104
scc                                22
actinic keratosis (ak)             18
pbk                                16
pigment benign keratosis (pbk)      7
ak                                  6
df                                  5
dermatofibroma                      2
Name: count, dtype: int64

In [28]:
df_payload["ground_truth"].value_counts()

ground_truth
Nevus                         136
Pigmented benign keratosis     40
Melanoma                       40
Basal cell carcinoma           30
Squamous cell carcinoma        23
Dermatofibroma                 16
Actinic Keratosis              15
Name: count, dtype: int64

In [30]:
label_mapping = {
    "nevus": "Nevus",
    "melanoma": "Melanoma",
    "pigment benign keratosis (pbk)": "Pigmented benign keratosis",
    "pbk": "Pigmented benign keratosis",
    "actinic keratosis (ak)": "Actinic Keratosis",
    "ak": "Actinic Keratosis",
    "basal cell carcinoma": "Basal cell carcinoma",
    "scc": "Squamous cell carcinoma",
    "df": "Dermatofibroma",
    "dermatofibroma": "Dermatofibroma"
}

def normalize_label(label):
    if isinstance(label, str):
        return label_mapping.get(label.strip(), label.strip())
    return label 

In [31]:
df_payload['diagnosis'] = df_payload['diagnosis'].apply(normalize_label)
df_payload['ground_truth'] = df_payload['ground_truth'].apply(normalize_label)

In [32]:
df_payload["diagnosis"].isna().sum()

np.int64(0)

In [33]:
df_payload[df_payload['diagnosis'].isna()]

,image_path,diagnosis,ground_truth,melanocytic_gt,confidence,differential,support,warning,recommendation


In [34]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

y_true = df_payload["ground_truth"]
y_pred = df_payload["diagnosis"]

In [35]:
# --- Accuracy ---
acc = accuracy_score(y_true, y_pred)
print(f"✅ Overall Accuracy: {acc:.2f}\n")

# --- Precision, Recall, F1-Score ---
# This report is the most useful output!
report = classification_report(y_true, y_pred)
print("📊 Classification Report:\n", report)

✅ Overall Accuracy: 0.30

📊 Classification Report:
                             precision    recall  f1-score   support

         Actinic Keratosis       0.00      0.00      0.00        15
      Basal cell carcinoma       0.00      0.00      0.00        30
            Dermatofibroma       0.71      0.31      0.43        16
                  Melanoma       0.18      0.55      0.28        40
                     Nevus       0.56      0.43      0.48       136
Pigmented benign keratosis       0.13      0.07      0.10        40
   Squamous cell carcinoma       0.09      0.09      0.09        23

                  accuracy                           0.30       300
                 macro avg       0.24      0.21      0.20       300
              weighted avg       0.34      0.30      0.30       300



/Users/thinakonelouangdy/Code/Thesis/Code/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/thinakonelouangdy/Code/Thesis/Code/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/thinakonelouangdy/Code/Thesis/Code/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _

In [36]:
df_payload[df_payload["diagnosis"] == "Nevus"]

,image_path,diagnosis,ground_truth,melanocytic_gt,confidence,differential,support,warning,recommendation
2,ISIC_0030535,Nevus,Nevus,True,0.85,"[{'label': 'melanoma', 'reason': 'multiple sha...",[Symmetric shape and symmetric color distribut...,[No single high-specificity melanocytic anchor...,Consider conservative management with clinical...
4,ISIC_0024944,Nevus,Nevus,True,0.85,"[{'label': 'melanoma', 'reason': 'Irregular pa...",[Symmetric shape and symmetric color distribut...,[No classic regular pigment network centrally ...,For informational use: consider short-term pho...
6,ISIC_0031540,Nevus,Nevus,True,0.45,"[{'label': 'melanoma', 'reason': 'central dark...",[Asymmetric shape and asymmetric color distrib...,[No strong melanocytic anchors (no pigment net...,Lesion is indeterminate on dermoscopy. Conside...
7,ISIC_0028709,Nevus,Melanoma,True,0.25,"[{'label': 'melanoma', 'reason': 'Asymmetry, m...","[Central amorphous brown blotch present, Few c...",[No strong melanocytic anchors (no pigment net...,Consider dermatologic review for clinical corr...
11,ISIC_0028652,Nevus,Basal cell carcinoma,False,0.55,"[{'diagnosis': 'melanoma', 'reason': 'Asymmetr...","[Asymmetric shape and color, Multicomponent pa...",[No high-specificity melanocytic anchors prese...,Consider clinical correlation and dermatology ...
...,...,...,...,...,...,...,...,...,...
286,ISIC_0025671,Nevus,Nevus,True,0.70,"[{'diagnosis': 'melanoma', 'reason': 'Atypical...",[Central irregular/atypical pigment network pr...,[Streaks/radial streaming plus atypical networ...,Consider dermatologic assessment for clinical ...
287,ISIC_0033919,Nevus,Nevus,True,0.85,"[{'label': 'melanoma', 'reason': 'Small darker...","[Symmetric shape and symmetric color, Well-dem...",[No high-specificity melanocytic anchors prese...,Consider short-term dermoscopic follow-up (e.g...
289,ISIC_0025451,Nevus,Melanoma,True,0.65,"[{'diagnosis': 'melanoma', 'reason': 'Asymmetr...",[Coarse brown granules/dots present both centr...,[No high-specificity melanocytic anchors (typi...,Consider specialist dermatology review and low...
290,ISIC_0028579,Nevus,Melanoma,True,0.55,"[{'diagnosis': 'melanoma', 'reason': 'Asymmetr...","[Asymmetric shape and color, Multicomponent pa...",[No high-specificity melanoma anchors present ...,Lesion appears melanocytic with some atypical ...
